# Unified Risk Management Pipeline
## Empirical Calibration · ML Volatility Forecasting · Monte Carlo Simulation · Regime-Conditional VaR/ES

**Architecture:**
```
Real CRSP Data (AAPL / C / F, 2001–2023)
        │
        ▼
Feature Engineering  ──────────────────────────────────────────────────────────────────────────────────────
(Log Returns, Rolling Vol, MACD, RSI, Macro: CPI/UNRATE/GDP)
        │
        ▼
Regime Detection (Threshold + Markov Switching)
        │
        ├──► ML Volatility Forecasting (GBM · LSTM · GRU)  ──► σ̂ per regime
        │                                                         │
        ▼                                                         ▼
Empirical Calibration ◄──────────────────────────────── Heston / GBM parameter grid
(μ, σ, κ, θ, ξ, ρ per ticker per regime)
        │
        ▼
Monte Carlo Simulation
(GBM · Heston · Correlated Multi-Asset via Cholesky)
        │
        ▼
Risk Metrics per Regime
(VaR₉₅ · VaR₉₉ · ES₉₅ · ES₉₉ · Sortino · Max Drawdown)
        │
        ▼
Credit & Operational Risk
(Merton Default Probability · Counterparty CVA · LDA Operational Loss)
        │
        ▼
Portfolio Optimisation
(Mean-Variance · Risk Parity · Stress Testing)
```


## 0. Imports & Configuration

In [ ]:

# ── Standard ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# ── Scientific / Stats ────────────────────────────────────────────────────────
from scipy.stats import norm, poisson, expon
from scipy.optimize import minimize
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

# ── Sklearn ───────────────────────────────────────────────────────────────────
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ── Deep Learning ─────────────────────────────────────────────────────────────
try:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    KERAS_AVAILABLE = True
except ImportError:
    KERAS_AVAILABLE = False
    print("TensorFlow not available – LSTM/GRU sections will be skipped.")

# ── Plot style ─────────────────────────────────────────────────────────────────
sns.set_style("darkgrid")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
COLORS = {"AAPL": "#1f77b4", "C": "#ff7f0e", "F": "#2ca02c"}
REGIME_COLORS = {"Bull": "#2ca02c", "Bear": "#d62728", "Crisis": "#ff7f0e"}

np.random.seed(42)
print("✓ All imports loaded.")


## 1. Data Loading & Initial Inspection

In [ ]:

# ── Load dataset (PROJECT_DATASET.csv – CRSP + macro, 2001-2023) ──────────────
# Update path if running locally
DATA_PATH = "PROJECT_DATASET.csv"

data = pd.read_csv(DATA_PATH)
data = data.drop(columns=["Unnamed: 0"], errors="ignore")
data["date"] = pd.to_datetime(data["date"])
data = data.sort_values(["TICKER", "date"]).reset_index(drop=True)

# Drop the 4 missing VOL rows
data = data.dropna(subset=["VOL"])

print(f"Dataset: {data.shape[0]:,} rows × {data.shape[1]} columns")
print(f"Tickers : {data['TICKER'].unique()}")
print(f"Period  : {data['date'].min().date()} → {data['date'].max().date()}")
print(f"Columns : {data.columns.tolist()}")
data.head()


## 2. Feature Engineering

In [ ]:

# ── Log Returns ────────────────────────────────────────────────────────────────
data["LogReturn"] = data.groupby("TICKER")["PRC"].transform(
    lambda x: np.log(x / x.shift(1))
)

# ── Rolling Volatility (20-day) ────────────────────────────────────────────────
data["Volatility"] = data.groupby("TICKER")["RET"].transform(
    lambda x: x.rolling(window=20).std()
)

# ── Moving Averages ────────────────────────────────────────────────────────────
data["MA_20"] = data.groupby("TICKER")["PRC"].transform(lambda x: x.rolling(20).mean())
data["MA_50"] = data.groupby("TICKER")["PRC"].transform(lambda x: x.rolling(50).mean())

# ── MACD ───────────────────────────────────────────────────────────────────────
data["EMA_12"] = data.groupby("TICKER")["PRC"].transform(
    lambda x: x.ewm(span=12, adjust=False).mean()
)
data["EMA_26"] = data.groupby("TICKER")["PRC"].transform(
    lambda x: x.ewm(span=26, adjust=False).mean()
)
data["MACD"] = data["EMA_12"] - data["EMA_26"]

# ── RSI (14-day) ───────────────────────────────────────────────────────────────
delta = data.groupby("TICKER")["PRC"].diff()
gain  = delta.clip(lower=0).groupby(data["TICKER"]).transform(lambda x: x.rolling(14).mean())
loss  = (-delta.clip(upper=0)).groupby(data["TICKER"]).transform(lambda x: x.rolling(14).mean())
data["RSI"] = 100 - (100 / (1 + gain / loss.replace(0, np.nan)))

# ── Bid-Ask Spread ─────────────────────────────────────────────────────────────
data["Spread"] = data["ASK"] - data["BID"]

# ── Lagged Returns & Macro ─────────────────────────────────────────────────────
for lag in range(1, 4):
    data[f"RET_Lag{lag}"] = data.groupby("TICKER")["RET"].transform(lambda x: x.shift(lag))
    data[f"cpiret_Lag{lag}"] = data["cpiret"].shift(lag)
    data[f"UNRATE_Lag{lag}"] = data["UNRATE"].shift(lag)

# ── Standardise Macro Variables ────────────────────────────────────────────────
scaler_macro = StandardScaler()
data[["cpiret_norm", "UNRATE_norm", "GDP_norm"]] = scaler_macro.fit_transform(
    data[["cpiret", "UNRATE", "GDP"]]
)

# ── Interaction Features ───────────────────────────────────────────────────────
data["RET_cpiret"]  = data["RET"] * data["cpiret"]
data["RET_UNRATE"]  = data["RET"] * data["UNRATE"]

# ── Cumulative 30-day Return (for regime classification) ───────────────────────
data["CumRet30"] = data.groupby("TICKER")["RET"].transform(lambda x: x.rolling(30).sum())

data = data.dropna().reset_index(drop=True)
print(f"After feature engineering: {data.shape[0]:,} rows")
data.describe().round(4)


## 3. Market Regime Detection

In [ ]:

# ── 3a. Threshold-Based Regime ────────────────────────────────────────────────
# Bull  : daily return > +2%
# Bear  : daily return < -2%
# Crisis: in between  (consolidation / mean-reversion)
data["Regime_Threshold"] = np.where(
    data["RET"] >  0.02, "Bull",
    np.where(data["RET"] < -0.02, "Bear", "Crisis")
)

print("Threshold regime distribution:")
print(data["Regime_Threshold"].value_counts())


In [ ]:

# ── 3b. Markov Switching Model (fitted on pooled returns) ─────────────────────
print("Fitting Markov Switching model (3 regimes)...")
markov_model = MarkovRegression(
    data["RET"], k_regimes=3, trend="c", switching_variance=True
)
markov_results = markov_model.fit(disp=False)
data["Regime_Markov_raw"] = markov_results.predict()

# Map Markov states to meaningful labels by average return
state_means = {}
for s in range(3):
    mask = data["Regime_Markov_raw"].round(0).astype(int) == s
    state_means[s] = data.loc[mask, "RET"].mean()

sorted_states = sorted(state_means, key=lambda s: state_means[s])
label_map = {sorted_states[0]: "Bear", sorted_states[1]: "Crisis", sorted_states[2]: "Bull"}
data["Regime_Markov"] = data["Regime_Markov_raw"].round(0).astype(int).map(label_map)

print("\nMarkov regime distribution:")
print(data["Regime_Markov"].value_counts())


In [ ]:

# ── 3c. Visualise Regime Classification ──────────────────────────────────────
ticker = "AAPL"
td = data[data["TICKER"] == ticker].copy()

fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

# Price
axes[0].plot(td["date"], td["PRC"], color=COLORS[ticker], lw=1.2)
axes[0].set_ylabel("Price ($)")
axes[0].set_title(f"{ticker} – Price, Threshold Regime & Markov Regime")

# Threshold regime overlay
for regime, color in REGIME_COLORS.items():
    mask = td["Regime_Threshold"] == regime
    axes[1].scatter(td.loc[mask, "date"], td.loc[mask, "RET"],
                    s=2, color=color, label=regime, alpha=0.6)
axes[1].set_ylabel("Daily Return")
axes[1].axhline(0, color="black", lw=0.8)
axes[1].legend(markerscale=4, loc="upper left")
axes[1].set_title("Threshold Regime")

# Markov smoothed probability
axes[2].plot(td["date"], td["Regime_Markov_raw"], color="#9467bd", lw=1.2)
axes[2].set_ylabel("Markov State")
axes[2].set_title("Markov Switching State (0=Bear, 1=Crisis, 2=Bull)")

plt.tight_layout()
plt.show()


## 4. Empirical Parameter Calibration (per Ticker × Regime)

In [ ]:

def calibrate_gbm_params(ret_series):
    """Calibrate annualised GBM drift and volatility from log-return series."""
    mu    = ret_series.mean() * 252          # annualised drift
    sigma = ret_series.std()  * np.sqrt(252)  # annualised vol
    return mu, sigma

def calibrate_heston_params(ret_series, vol_series):
    """
    Fit Heston parameters via moment-matching on the realised variance process.
    Uses: κ (mean-reversion speed), θ (long-run variance), ξ (vol-of-vol), ρ (price-vol corr).
    """
    rv = vol_series**2  # realised variance (daily)
    
    theta = rv.mean()                # long-run variance
    # κ estimated from AR(1) on variance (discrete Ornstein-Uhlenbeck)
    rv_lag = rv.shift(1).dropna()
    rv_t   = rv.iloc[1:]
    kappa_dt = 1 - np.polyfit(rv_lag, rv_t, 1)[0]
    kappa = max(kappa_dt * 252, 0.1)  # annualise, floor at 0.1
    xi    = rv.std() * np.sqrt(252)   # annualised vol-of-vol
    # ρ: correlation between price changes and variance changes
    rho   = np.corrcoef(ret_series.iloc[1:], rv.diff().dropna())[0, 1]
    rho   = np.clip(rho, -0.99, 0.99)
    return kappa, theta, xi, rho

# ── Build calibration table ────────────────────────────────────────────────────
calib_records = []

for ticker in ["AAPL", "C", "F"]:
    td = data[data["TICKER"] == ticker]
    
    # Full-sample calibration
    mu_full, sigma_full = calibrate_gbm_params(td["RET"])
    kappa, theta, xi, rho = calibrate_heston_params(td["RET"], td["Volatility"])
    calib_records.append({
        "Ticker": ticker, "Regime": "Full",
        "mu": round(mu_full, 4), "sigma": round(sigma_full, 4),
        "kappa": round(kappa, 4), "theta": round(theta, 6),
        "xi": round(xi, 4), "rho": round(rho, 4)
    })
    
    # Per-regime calibration
    for regime in ["Bull", "Bear", "Crisis"]:
        rd = td[td["Regime_Threshold"] == regime]
        if len(rd) < 30:
            continue
        mu_r, sigma_r = calibrate_gbm_params(rd["RET"])
        k_r, t_r, xi_r, rho_r = calibrate_heston_params(rd["RET"], rd["Volatility"])
        calib_records.append({
            "Ticker": ticker, "Regime": regime,
            "mu": round(mu_r, 4), "sigma": round(sigma_r, 4),
            "kappa": round(k_r, 4), "theta": round(t_r, 6),
            "xi": round(xi_r, 4), "rho": round(rho_r, 4)
        })

calib_df = pd.DataFrame(calib_records)
print("Empirical calibration results:")
print(calib_df.to_string(index=False))


## 5. ML Volatility Forecasting (GBM · RF · LSTM · GRU)

In [ ]:

# ── Feature set (with and without macro) ──────────────────────────────────────
MACRO_FEATURES = ["cpiret_norm", "UNRATE_norm", "GDP_norm",
                  "cpiret_Lag1", "UNRATE_Lag1", "cpiret_Lag2", "UNRATE_Lag2"]
BASE_FEATURES  = ["RET", "RET_Lag1", "RET_Lag2", "RET_Lag3",
                  "Volatility", "MACD", "RSI", "Spread", "MA_20", "MA_50",
                  "RET_cpiret", "RET_UNRATE"]
ALL_FEATURES   = BASE_FEATURES + MACRO_FEATURES

TARGET = "Volatility"

def evaluate_metrics(y_true, y_pred):
    """Returns dict of regression metrics including VaR/ES on prediction errors."""
    n, p = len(y_true), 1
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1e-8, y_true))) * 100
    r2   = r2_score(y_true, y_pred)
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    return dict(MSE=mse, RMSE=rmse, MAE=mae, MAPE=mape, R2=r2, AdjR2=adj_r2)

# ── Train per ticker – GBM and RF only (LSTM/GRU gated behind KERAS_AVAILABLE) ─
ml_results    = {}   # {ticker: {model: {regime: metrics}}}
vol_forecasts = {}   # {ticker: {model: pd.Series of predicted vol}}

N_SPLITS = 3
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

for ticker in ["AAPL", "C", "F"]:
    print(f"\n{'─'*50}\nTicker: {ticker}")
    td = data[data["TICKER"] == ticker].copy().reset_index(drop=True)
    
    ml_results[ticker]    = {}
    vol_forecasts[ticker] = {}
    
    for model_name, ModelClass, params in [
        ("GBM", GradientBoostingRegressor, {"n_estimators": 200, "max_depth": 4, "learning_rate": 0.05}),
        ("RF",  RandomForestRegressor,     {"n_estimators": 200, "max_depth": 6, "min_samples_leaf": 5}),
    ]:
        print(f"  Training {model_name}...")
        fold_metrics = []
        all_preds    = np.full(len(td), np.nan)
        
        for fold, (tr_idx, te_idx) in enumerate(tscv.split(td)):
            X_tr = td.iloc[tr_idx][ALL_FEATURES].fillna(0)
            X_te = td.iloc[te_idx][ALL_FEATURES].fillna(0)
            y_tr = td.iloc[tr_idx][TARGET]
            y_te = td.iloc[te_idx][TARGET]
            
            sc = StandardScaler()
            X_tr_s = sc.fit_transform(X_tr)
            X_te_s = sc.transform(X_te)
            
            m = ModelClass(**params, random_state=42)
            m.fit(X_tr_s, y_tr)
            preds = m.predict(X_te_s)
            
            all_preds[te_idx] = preds
            fold_metrics.append(evaluate_metrics(y_te.values, preds))
        
        avg_metrics = {k: np.mean([fm[k] for fm in fold_metrics]) for k in fold_metrics[0]}
        ml_results[ticker][model_name] = avg_metrics
        vol_forecasts[ticker][model_name] = pd.Series(all_preds, index=td.index)
        
        print(f"    RMSE={avg_metrics['RMSE']:.5f}  R²={avg_metrics['R2']:.4f}  MAPE={avg_metrics['MAPE']:.2f}%")

print("\n✓ ML training complete.")


In [ ]:

# ── Optional: LSTM / GRU ──────────────────────────────────────────────────────
if KERAS_AVAILABLE:
    SEQ_LEN = 20
    
    def build_model(model_type, seq_len, n_features):
        """Build LSTM or GRU sequential model."""
        m = Sequential()
        Layer = LSTM if model_type == "LSTM" else GRU
        m.add(Layer(64, activation="tanh", input_shape=(seq_len, n_features), return_sequences=True))
        m.add(Dropout(0.2))
        m.add(Layer(32, activation="tanh"))
        m.add(Dense(1))
        m.compile(optimizer="adam", loss="mse")
        return m

    def make_sequences(X, y, seq_len):
        """Convert tabular data into (samples, seq_len, features) sequences."""
        Xs, ys = [], []
        for i in range(seq_len, len(X)):
            Xs.append(X[i-seq_len:i])
            ys.append(y[i])
        return np.array(Xs), np.array(ys)

    es = EarlyStopping(patience=5, restore_best_weights=True)

    for ticker in ["AAPL", "C", "F"]:
        print(f"\n{ticker} – LSTM/GRU")
        td = data[data["TICKER"] == ticker].copy().reset_index(drop=True)
        sc = StandardScaler()
        X_all = sc.fit_transform(td[ALL_FEATURES].fillna(0))
        y_all = td[TARGET].values
        
        for model_type in ["LSTM", "GRU"]:
            fold_metrics, all_preds = [], np.full(len(td), np.nan)
            
            for fold, (tr_idx, te_idx) in enumerate(tscv.split(td)):
                X_tr_s, X_te_s = X_all[tr_idx], X_all[te_idx]
                y_tr, y_te = y_all[tr_idx], y_all[te_idx]
                
                X_tr_seq, y_tr_seq = make_sequences(X_tr_s, y_tr, SEQ_LEN)
                X_te_seq, y_te_seq = make_sequences(X_te_s, y_te, SEQ_LEN)
                
                model = build_model(model_type, SEQ_LEN, len(ALL_FEATURES))
                model.fit(X_tr_seq, y_tr_seq, epochs=30, batch_size=32,
                          validation_split=0.1, callbacks=[es], verbose=0)
                preds = model.predict(X_te_seq, verbose=0).flatten()
                
                valid_te = te_idx[SEQ_LEN:]
                if len(valid_te) == len(preds):
                    all_preds[valid_te] = preds
                fold_metrics.append(evaluate_metrics(y_te_seq, preds))
            
            avg_metrics = {k: np.mean([fm[k] for fm in fold_metrics]) for k in fold_metrics[0]}
            ml_results[ticker][model_type] = avg_metrics
            vol_forecasts[ticker][model_type] = pd.Series(all_preds, index=td.index)
            print(f"  {model_type}: RMSE={avg_metrics['RMSE']:.5f}  R²={avg_metrics['R2']:.4f}")
    
    print("\n✓ LSTM/GRU training complete.")
else:
    print("Skipping LSTM/GRU (TensorFlow not installed).")


In [ ]:

# ── ML Results Summary Table ──────────────────────────────────────────────────
rows = []
for ticker in ml_results:
    for model, metrics in ml_results[ticker].items():
        rows.append({"Ticker": ticker, "Model": model, **{k: round(v, 5) for k, v in metrics.items()}})

ml_summary = pd.DataFrame(rows)
print("ML Volatility Forecasting Summary:")
print(ml_summary[["Ticker", "Model", "RMSE", "MAE", "R2", "MAPE"]].to_string(index=False))


In [ ]:

# ── Forecast vs Actual plot ───────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=False)

for ax, ticker in zip(axes, ["AAPL", "C", "F"]):
    td = data[data["TICKER"] == ticker].reset_index(drop=True)
    ax.plot(td["date"], td["Volatility"], color="black", lw=1.2, label="Actual Vol", zorder=3)
    
    palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#9467bd"]
    for (model_name, preds), color in zip(vol_forecasts[ticker].items(), palette):
        valid = ~preds.isna()
        ax.plot(td.loc[valid, "date"], preds[valid], color=color, lw=0.9,
                alpha=0.75, label=f"{model_name} forecast")
    
    ax.set_title(f"{ticker} – Rolling Volatility: Actual vs ML Forecasts")
    ax.set_ylabel("Volatility (daily)")
    ax.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()


## 6. Monte Carlo Simulation — Empirically Calibrated

In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# 6a. GBM simulation (calibrated μ, σ per ticker per regime)
# ─────────────────────────────────────────────────────────────────────────────
def simulate_gbm(S0, mu, sigma, T=1.0, n_sim=10_000, n_steps=252):
    """Geometric Brownian Motion: returns (n_steps+1, n_sim) price matrix."""
    dt = T / n_steps
    paths = np.zeros((n_steps + 1, n_sim))
    paths[0] = S0
    for t in range(1, n_steps + 1):
        Z = np.random.standard_normal(n_sim)
        paths[t] = paths[t-1] * np.exp((mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z)
    return paths


# ─────────────────────────────────────────────────────────────────────────────
# 6b. Heston model (calibrated κ, θ, ξ, ρ per ticker)
# ─────────────────────────────────────────────────────────────────────────────
def simulate_heston(S0, V0, mu, kappa, theta, xi, rho, T=1.0, n_sim=10_000, n_steps=252):
    """Full-truncation Heston with Euler discretisation. Returns (S paths, V paths)."""
    dt = T / n_steps
    S = np.zeros((n_steps + 1, n_sim))
    V = np.zeros((n_steps + 1, n_sim))
    S[0], V[0] = S0, V0
    for t in range(1, n_steps + 1):
        Z1 = np.random.standard_normal(n_sim)
        Z2 = rho * Z1 + np.sqrt(1 - rho**2) * np.random.standard_normal(n_sim)
        V[t] = np.maximum(
            V[t-1] + kappa * (theta - V[t-1]) * dt + xi * np.sqrt(np.maximum(V[t-1], 0) * dt) * Z1,
            0  # full truncation
        )
        S[t] = S[t-1] * np.exp((mu - 0.5 * V[t-1]) * dt + np.sqrt(np.maximum(V[t-1], 0) * dt) * Z2)
    return S, V


# ─────────────────────────────────────────────────────────────────────────────
# 6c. Correlated multi-asset GBM (Cholesky decomposition)
# ─────────────────────────────────────────────────────────────────────────────
def simulate_correlated_assets(S0_vec, mu_vec, sigma_vec, corr_matrix,
                                T=1.0, n_sim=5_000, n_steps=252):
    """
    Multi-asset correlated GBM.
    Returns (n_steps+1, n_assets, n_sim) array.
    """
    dt = T / n_steps
    n_assets = len(S0_vec)
    paths = np.zeros((n_steps + 1, n_assets, n_sim))
    paths[0] = np.array(S0_vec)[:, np.newaxis]
    L = np.linalg.cholesky(corr_matrix)
    for t in range(1, n_steps + 1):
        Z = np.random.standard_normal((n_assets, n_sim))
        corr_Z = L @ Z
        for i in range(n_assets):
            paths[t, i] = paths[t-1, i] * np.exp(
                (mu_vec[i] - 0.5 * sigma_vec[i]**2) * dt
                + sigma_vec[i] * np.sqrt(dt) * corr_Z[i]
            )
    return paths

print("✓ Simulation functions defined.")


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# 6d. Run simulations for each ticker using empirically calibrated params
# ─────────────────────────────────────────────────────────────────────────────
N_SIM   = 10_000
N_STEPS = 252
T       = 1.0

simulation_results = {}  # {ticker: {"GBM": paths, "Heston": (S, V)}}

for ticker in ["AAPL", "C", "F"]:
    row_full = calib_df[(calib_df["Ticker"] == ticker) & (calib_df["Regime"] == "Full")].iloc[0]
    td = data[data["TICKER"] == ticker]
    S0 = float(td["PRC"].iloc[-1])     # last observed price as starting point
    V0 = float(td["Volatility"].iloc[-1])**2  # last realised variance

    # GBM
    gbm_paths = simulate_gbm(S0, row_full["mu"], row_full["sigma"],
                              T, N_SIM, N_STEPS)
    
    # Heston
    heston_S, heston_V = simulate_heston(
        S0, V0, row_full["mu"],
        row_full["kappa"], row_full["theta"],
        row_full["xi"], row_full["rho"],
        T, N_SIM, N_STEPS
    )
    
    simulation_results[ticker] = {"GBM": gbm_paths, "Heston_S": heston_S, "Heston_V": heston_V}
    print(f"{ticker}: S0={S0:.2f}  μ={row_full['mu']:.4f}  σ={row_full['sigma']:.4f}  "
          f"κ={row_full['kappa']:.3f}  ξ={row_full['xi']:.4f}  ρ={row_full['rho']:.3f}")

print("\n✓ Simulations complete.")


In [ ]:

# ── Plot: GBM vs Heston paths for AAPL ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
t_axis = np.linspace(0, 1, N_STEPS + 1)

for ax, model_key, title in zip(
    axes,
    ["GBM", "Heston_S"],
    ["AAPL – GBM (Calibrated)", "AAPL – Heston (Calibrated)"]
):
    paths = simulation_results["AAPL"][model_key]
    ax.plot(t_axis, paths[:, :150], lw=0.4, alpha=0.25, color="#1f77b4")
    ax.plot(t_axis, np.percentile(paths, 5, axis=1), "r--", lw=1.5, label="5th pctl")
    ax.plot(t_axis, np.percentile(paths, 50, axis=1), "k-", lw=1.5, label="Median")
    ax.plot(t_axis, np.percentile(paths, 95, axis=1), "g--", lw=1.5, label="95th pctl")
    ax.set_title(title)
    ax.set_xlabel("Time (years)")
    ax.set_ylabel("Price ($)")
    ax.legend()

plt.tight_layout()
plt.show()


## 7. Risk Metrics — Simulation-Based VaR & ES

In [ ]:

def compute_risk_metrics(paths, S0, confidence_levels=(0.95, 0.99)):
    """
    Compute VaR and ES from terminal price distribution.
    Returns losses (negative P&L), VaR and ES at each confidence level.
    """
    terminal = paths[-1]
    pnl      = terminal - S0          # dollar P&L
    losses   = -pnl                   # losses are positive

    metrics = {}
    for alpha in confidence_levels:
        var = np.percentile(losses, alpha * 100)
        es  = losses[losses >= var].mean()
        metrics[f"VaR_{int(alpha*100)}"] = round(var, 4)
        metrics[f"ES_{int(alpha*100)}"]  = round(es, 4)
    
    metrics["E[P&L]"]   = round(pnl.mean(), 4)
    metrics["Vol(P&L)"] = round(pnl.std(), 4)
    return metrics

# ── Risk metrics table ─────────────────────────────────────────────────────────
risk_rows = []
for ticker in ["AAPL", "C", "F"]:
    td = data[data["TICKER"] == ticker]
    S0 = float(td["PRC"].iloc[-1])
    
    for model_key, label in [("GBM", "GBM"), ("Heston_S", "Heston")]:
        paths = simulation_results[ticker][model_key]
        m = compute_risk_metrics(paths, S0)
        risk_rows.append({"Ticker": ticker, "Model": label, "S0": round(S0, 2), **m})

risk_df = pd.DataFrame(risk_rows)
print("Simulation-Based Risk Metrics:")
print(risk_df.to_string(index=False))


In [ ]:

# ── Additional portfolio metrics ───────────────────────────────────────────────
def sortino_ratio(returns, rf=0.03/252):
    """Daily Sortino ratio."""
    excess = returns - rf
    downside = returns[returns < 0].std()
    return (excess.mean() / downside) * np.sqrt(252) if downside > 0 else np.nan

def max_drawdown(returns):
    """Maximum drawdown from return series."""
    cum = (1 + returns).cumprod()
    peak = cum.expanding().max()
    dd = (cum - peak) / peak
    return dd.min()

print("Portfolio risk metrics (empirical, full sample):")
print(f"{'Ticker':<8} {'Sortino':>10} {'MaxDD':>10}")
for ticker in ["AAPL", "C", "F"]:
    ret = data[data["TICKER"] == ticker]["RET"]
    print(f"{ticker:<8} {sortino_ratio(ret):>10.3f} {max_drawdown(ret):>10.4%}")


## 8. Regime-Conditional Risk — VaR/ES per Market State

In [ ]:

# Simulate paths using regime-specific calibrated parameters
regime_risk_rows = []

for ticker in ["AAPL", "C", "F"]:
    td = data[data["TICKER"] == ticker]
    S0 = float(td["PRC"].iloc[-1])
    
    for regime in ["Bull", "Bear", "Crisis"]:
        row = calib_df[(calib_df["Ticker"] == ticker) & (calib_df["Regime"] == regime)]
        if row.empty:
            continue
        row = row.iloc[0]
        
        # GBM with regime-specific params
        paths = simulate_gbm(S0, row["mu"], row["sigma"],
                             T=0.25, n_sim=N_SIM, n_steps=63)  # 1-quarter horizon
        m = compute_risk_metrics(paths, S0)
        regime_risk_rows.append({
            "Ticker": ticker, "Regime": regime, **m,
            "mu": row["mu"], "sigma": row["sigma"]
        })

regime_risk_df = pd.DataFrame(regime_risk_rows)
print("Regime-Conditional VaR/ES (1-quarter horizon, GBM):")
print(regime_risk_df[["Ticker","Regime","mu","sigma","VaR_95","ES_95","VaR_99","ES_99"]].to_string(index=False))


In [ ]:

# ── Heatmap: VaR_95 across tickers and regimes ─────────────────────────────────
pivot = regime_risk_df.pivot(index="Ticker", columns="Regime", values="VaR_95")
pivot = pivot[["Bull", "Crisis", "Bear"]]  # logical order

plt.figure(figsize=(9, 4))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="RdYlGn_r",
            linewidths=0.5, cbar_kws={"label": "VaR 95% ($)"})
plt.title("1-Quarter VaR₉₅ by Ticker and Market Regime (GBM, $)")
plt.tight_layout()
plt.show()


## 9. Multi-Asset Portfolio — Correlated Simulation & Optimisation

In [ ]:

# ── Estimate empirical correlation matrix (AAPL / C / F) ──────────────────────
pivot_ret = data.pivot(index="date", columns="TICKER", values="RET")[["AAPL", "C", "F"]].dropna()
emp_corr  = pivot_ret.corr().values

print("Empirical return correlation matrix:")
print(pd.DataFrame(emp_corr, index=["AAPL","C","F"], columns=["AAPL","C","F"]).round(3))

# Last prices as starting values
S0_vec    = [float(data[data["TICKER"]==t]["PRC"].iloc[-1]) for t in ["AAPL","C","F"]]
mu_vec    = [calib_df[(calib_df["Ticker"]==t)&(calib_df["Regime"]=="Full")]["mu"].values[0]
             for t in ["AAPL","C","F"]]
sigma_vec = [calib_df[(calib_df["Ticker"]==t)&(calib_df["Regime"]=="Full")]["sigma"].values[0]
             for t in ["AAPL","C","F"]]

# Run correlated multi-asset simulation
portfolio_paths = simulate_correlated_assets(
    S0_vec, mu_vec, sigma_vec, emp_corr,
    T=1.0, n_sim=5_000, n_steps=252
)
print(f"\nPortfolio paths shape: {portfolio_paths.shape}  (steps, assets, sims)")


In [ ]:

# ── Portfolio Optimisation ────────────────────────────────────────────────────
def mean_variance_optimise(returns_df, rf=0.03):
    """Maximise Sharpe ratio using mean-variance optimisation."""
    mu_vec   = returns_df.mean().values * 252
    cov_mat  = returns_df.cov().values * 252
    n        = len(mu_vec)

    def neg_sharpe(w):
        ret  = w @ mu_vec
        vol  = np.sqrt(w @ cov_mat @ w)
        return -(ret - rf) / vol if vol > 0 else 0

    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]
    bounds      = [(0, 1)] * n
    w0          = np.ones(n) / n

    result = minimize(neg_sharpe, w0, method="SLSQP",
                      bounds=bounds, constraints=constraints)
    return result.x

def risk_parity_optimise(returns_df):
    """Equal risk contribution (risk parity)."""
    cov = returns_df.cov().values * 252
    n   = cov.shape[0]

    def rc_objective(w):
        port_var = w @ cov @ w
        rc = (cov @ w) * w / port_var
        return np.sum((rc - port_var / n)**2)

    result = minimize(rc_objective, np.ones(n)/n, method="SLSQP",
                      bounds=[(0.01, 0.99)]*n,
                      constraints=[{"type": "eq", "fun": lambda w: np.sum(w)-1}])
    return result.x

w_mv = mean_variance_optimise(pivot_ret)
w_rp = risk_parity_optimise(pivot_ret)
w_ew = np.array([1/3, 1/3, 1/3])

print("Portfolio Weights:")
print(f"{'Strategy':<20} {'AAPL':>8} {'C':>8} {'F':>8}")
for name, w in [("Mean-Variance", w_mv), ("Risk Parity", w_rp), ("Equal Weight", w_ew)]:
    print(f"{name:<20} {w[0]:>8.3f} {w[1]:>8.3f} {w[2]:>8.3f}")


In [ ]:

# ── Portfolio VaR/ES from correlated simulation ────────────────────────────────
print("Portfolio Risk Metrics (1-year horizon):")
print(f"{'Strategy':<20} {'E[P&L]':>10} {'σ(P&L)':>10} {'VaR_95':>10} {'ES_95':>10} {'VaR_99':>10} {'ES_99':>10}")

for strategy_name, w in [("Mean-Variance", w_mv), ("Risk Parity", w_rp), ("Equal Weight", w_ew)]:
    # Portfolio terminal P&L = weighted sum of individual terminal P&Ls
    port_terminal = np.zeros(5_000)
    for i, ticker in enumerate(["AAPL", "C", "F"]):
        S0_i = S0_vec[i]
        pnl_i = portfolio_paths[-1, i, :] - S0_i
        port_terminal += w[i] * pnl_i
    
    losses = -port_terminal
    var95  = np.percentile(losses, 95)
    es95   = losses[losses >= var95].mean()
    var99  = np.percentile(losses, 99)
    es99   = losses[losses >= var99].mean()
    
    print(f"{strategy_name:<20} {port_terminal.mean():>10.2f} {port_terminal.std():>10.2f} "
          f"{var95:>10.2f} {es95:>10.2f} {var99:>10.2f} {es99:>10.2f}")


## 10. Credit Risk (Merton / CVA) & Operational Risk (LDA)

In [ ]:

# ── Merton Default Probability per ticker ─────────────────────────────────────
def merton_default_prob(S0, sigma_firm, D, r, T=1.0):
    """
    Merton model: probability of default = P(V_T < D).
    Uses BSM d2 formula. sigma_firm ≈ equity vol (simplification for unleveraged firms).
    """
    d2 = (np.log(S0 / D) + (r - 0.5 * sigma_firm**2) * T) / (sigma_firm * np.sqrt(T))
    return norm.cdf(-d2)

r_f = 0.05  # risk-free rate assumption

print("Merton Default Probabilities (D = 50% of S0 as proxy debt level):")
for ticker in ["AAPL", "C", "F"]:
    row  = calib_df[(calib_df["Ticker"]==ticker) & (calib_df["Regime"]=="Full")].iloc[0]
    S0_t = float(data[data["TICKER"]==ticker]["PRC"].iloc[-1])
    D    = S0_t * 0.50   # illustrative: debt = 50% of equity price
    pd_  = merton_default_prob(S0_t, row["sigma"], D, r_f)
    print(f"  {ticker}: S0={S0_t:.2f}  σ={row['sigma']:.4f}  D={D:.2f}  PD={pd_:.6f} ({pd_*100:.4f}%)")


In [ ]:

# ── Counterparty CVA ──────────────────────────────────────────────────────────
def counterparty_cva(exposure, PD, LGD, corr_matrix, n_sim=100_000):
    """
    Monte Carlo CVA: simulate correlated default events via Gaussian copula.
    Returns distribution of credit losses.
    """
    n = len(PD)
    Z = np.random.multivariate_normal(np.zeros(n), corr_matrix, n_sim)
    # Default threshold from PD (normal quantile)
    thresholds = norm.ppf(PD)
    defaults   = (Z < thresholds[np.newaxis, :]).astype(float)
    losses     = (defaults * np.array(exposure)[np.newaxis, :] * np.array(LGD)[np.newaxis, :]).sum(axis=1)
    return losses

# Illustrative: 3-counterparty portfolio (AAPL, C, F)
exposure_vec = [1_000_000, 2_000_000, 1_500_000]  # notional exposures
LGD_vec      = [0.40, 0.60, 0.50]                  # loss-given-default
PD_vec       = [0.005, 0.02, 0.015]                 # illustrative annual PDs

cva_losses = counterparty_cva(exposure_vec, PD_vec, LGD_vec, emp_corr)
cva_var95  = np.percentile(cva_losses, 95)
cva_es95   = cva_losses[cva_losses >= cva_var95].mean()
print(f"CVA Distribution (3-counterparty portfolio):")
print(f"  E[Loss]  = ${cva_losses.mean():>12,.0f}")
print(f"  VaR 95%  = ${cva_var95:>12,.0f}")
print(f"  ES  95%  = ${cva_es95:>12,.0f}")


In [ ]:

# ── LDA Operational Risk ──────────────────────────────────────────────────────
def lda_operational(event_freq, severity_scale, n_sim=100_000):
    """
    Loss Distribution Approach: compound Poisson / Exponential severity.
    Returns simulated total annual operational losses.
    """
    total = np.zeros(n_sim)
    for i in range(n_sim):
        n_events = np.random.poisson(event_freq)
        if n_events > 0:
            total[i] = np.sum(np.random.exponential(severity_scale, n_events))
    return total

op_losses = lda_operational(event_freq=50, severity_scale=20_000)
op_var99  = np.percentile(op_losses, 99)
op_es99   = op_losses[op_losses >= op_var99].mean()

print(f"LDA Operational Risk (λ=50 events/yr, E[severity]=$20k):")
print(f"  E[Loss]  = ${op_losses.mean():>12,.0f}")
print(f"  VaR 99%  = ${op_var99:>12,.0f}")
print(f"  ES  99%  = ${op_es99:>12,.0f}")

# Plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(op_losses, bins=100, color="#1f77b4", alpha=0.7, edgecolor="white")
ax.axvline(op_var99, color="red",    lw=1.8, linestyle="--", label=f"VaR 99%: ${op_var99:,.0f}")
ax.axvline(op_es99,  color="orange", lw=1.8, linestyle="--", label=f"ES 99%: ${op_es99:,.0f}")
ax.set_xlabel("Annual Operational Loss ($)")
ax.set_ylabel("Frequency")
ax.set_title("LDA – Operational Loss Distribution")
ax.legend()
plt.tight_layout()
plt.show()


## 11. Stress Testing & Scenario Analysis

In [ ]:

# ── Scenario definitions ───────────────────────────────────────────────────────
# Each scenario perturbs μ and σ relative to the calibrated full-sample values
SCENARIOS = {
    "Base Case":        {"mu_mult": 1.0,   "sigma_mult": 1.0},
    "2008 GFC":         {"mu_mult": -3.0,  "sigma_mult": 3.5},
    "COVID-19 (2020)":  {"mu_mult": -2.0,  "sigma_mult": 2.5},
    "Rate Shock (+200bp)": {"mu_mult": 0.5, "sigma_mult": 1.4},
    "Bull Run":         {"mu_mult": 2.0,   "sigma_mult": 0.8},
}

stress_rows = []
for ticker in ["AAPL", "C", "F"]:
    row = calib_df[(calib_df["Ticker"]==ticker) & (calib_df["Regime"]=="Full")].iloc[0]
    S0_t = float(data[data["TICKER"]==ticker]["PRC"].iloc[-1])
    
    for scenario, mults in SCENARIOS.items():
        mu_s    = row["mu"]    * mults["mu_mult"]
        sigma_s = row["sigma"] * mults["sigma_mult"]
        paths   = simulate_gbm(S0_t, mu_s, sigma_s, T=0.25, n_sim=10_000, n_steps=63)
        m       = compute_risk_metrics(paths, S0_t)
        stress_rows.append({"Ticker": ticker, "Scenario": scenario,
                             "mu_stressed": round(mu_s,4), "sigma_stressed": round(sigma_s,4),
                             **m})

stress_df = pd.DataFrame(stress_rows)
print("Stress Test Results (1-quarter VaR/ES per scenario):")
print(stress_df[["Ticker","Scenario","sigma_stressed","VaR_95","ES_95","VaR_99","ES_99"]]
      .to_string(index=False))


In [ ]:

# ── Stress test visualisation ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

for ax, ticker in zip(axes, ["AAPL", "C", "F"]):
    td_stress = stress_df[stress_df["Ticker"] == ticker]
    x = np.arange(len(SCENARIOS))
    width = 0.35
    
    ax.bar(x - width/2, td_stress["VaR_95"].values, width, label="VaR 95%", color="#1f77b4", alpha=0.85)
    ax.bar(x + width/2, td_stress["ES_95"].values,  width, label="ES 95%",  color="#d62728", alpha=0.85)
    
    ax.set_xticks(x)
    ax.set_xticklabels(list(SCENARIOS.keys()), rotation=30, ha="right", fontsize=8)
    ax.set_title(f"{ticker} – Scenario VaR₉₅ vs ES₉₅ (1Q, $)")
    ax.set_ylabel("Loss ($)")
    ax.legend(fontsize=8)
    ax.axhline(0, color="black", lw=0.8)

plt.tight_layout()
plt.show()


## 12. Consolidated Risk Summary Dashboard

In [ ]:

print("=" * 70)
print("UNIFIED RISK MANAGEMENT PIPELINE – CONSOLIDATED SUMMARY")
print("=" * 70)

print("\n[1] ML VOLATILITY FORECASTING (avg across CV folds)")
print(ml_summary[["Ticker","Model","RMSE","R2","MAPE"]].to_string(index=False))

print("\n[2] SIMULATION-BASED RISK METRICS (1-year, full sample)")
print(risk_df[["Ticker","Model","S0","VaR_95","ES_95","VaR_99","ES_99"]].to_string(index=False))

print("\n[3] REGIME-CONDITIONAL VaR₉₅ (1-quarter horizon)")
regime_pivot = regime_risk_df.pivot_table(
    index="Ticker", columns="Regime", values="VaR_95"
)[["Bull","Crisis","Bear"]]
print(regime_pivot.round(2).to_string())

print("\n[4] PORTFOLIO VaR/ES COMPARISON")
for strategy_name, w in [("Mean-Variance", w_mv), ("Risk Parity", w_rp), ("Equal Weight", w_ew)]:
    port_terminal = np.zeros(5_000)
    for i in range(3):
        pnl_i = portfolio_paths[-1, i, :] - S0_vec[i]
        port_terminal += w[i] * pnl_i
    losses = -port_terminal
    var95  = np.percentile(losses, 95)
    es95   = losses[losses >= var95].mean()
    print(f"  {strategy_name:<20}: VaR₉₅ = {var95:>8.2f}  ES₉₅ = {es95:>8.2f}")

print("\n[5] COUNTERPARTY CVA")
print(f"  Expected Loss = ${cva_losses.mean():>12,.0f}  |  VaR 95% = ${cva_var95:>12,.0f}")

print("\n[6] OPERATIONAL RISK (LDA)")
print(f"  Expected Loss = ${op_losses.mean():>12,.0f}  |  VaR 99% = ${op_var99:>12,.0f}")
print("=" * 70)
